In [1]:
pip install openai-clip

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 64.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.0 MB/s eta 0:00:00
  Created wheel for openai-clip: filename=openai_clip-1.0.1-py3-none-any.whl size=1368605 sha256=c7e45344314e1f799e8895e3bd54edbc3d8a142fd6859c0a865993ac0b83ed6d
  Stored in directory: /root/.cache/pip/wheels/ab/49/bc/c2342e8e14878210ba4825cf314a53f2570f6fb18b91fce3cf
Successfully built openai-clip


Part I: basic CLIP functionality with zero-shot predictions.

In [2]:
import torch
import clip
import shap
import json
import urllib.request
import numpy as np
from PIL import Image


Load the model ViT-B/32

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()


100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 294MiB/s]


CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): Sequential(
        (0): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): QuickGELU()
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
        (1): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          

Load datasets and class names

In [4]:
def load_clip(substring):
  # load model
  device = "cuda" if torch.cuda.is_available() else "cpu"
  model, preprocess = clip.load(substring, device=device)
  model.eval()

  # Get datasets
  X, y = shap.datasets.imagenet50()

  # Get class names
  url = "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json"
  with open(shap.datasets.cache(url)) as file:
      class_names = [v[1] for v in json.load(file).values()]

  # Specialized: convert class names to CLIP text prompts + encode
  text_prompts = [f"a photo of a {name.replace('_',' ')}" for name in class_names]
  text_tokens = clip.tokenize(text_prompts).to(device)

  with torch.no_grad():
      text_features = model.encode_text(text_tokens)
      text_features /= text_features.norm(dim=-1, keepdim=True)  # normalize

  # Define f and encode
  def f(X):
      tmp = []
      for i in range(len(X)):
          img = Image.fromarray(X[i].astype("uint8"))
          tmp.append(preprocess(img))

      X_tensor = torch.stack(tmp).to(device)

      with torch.no_grad():
          # Encode images
          image_features = model.encode_image(X_tensor)
          image_features /= image_features.norm(dim=-1, keepdim=True)

          # Similarity to all class text embeddings
          sim = image_features @ text_features.T  # shape [batch, num_classes]

      return sim

  return model, f, class_names, X

Plot SHAP values

In [5]:
def plot_shap(model, map, block_size):
  model, f, class_names, X = model

  # define a masker that is used to mask out partitions of the input image, this one uses a blurred background
  masker = shap.maskers.Image("inpaint_telea", X[0].shape)

  # By default the Partition explainer is used for all  partition explainer
  explainer = shap.Explainer(f, masker, output_names=class_names)

  # Shap values
  shap_values = explainer(X[:5], max_evals=500, batch_size=50, outputs=shap.Explanation.argsort.flip[:1])

  images = shap_values.data  # input images
  values = shap_values.values  # shap values
  classification = shap_values.output_names # outputs?
  # print(classification)

  i = 1
  img = images[i]
  sv_raw = values[i]
  out = classification[i]

  shap_min = np.min(sv_raw)
  shap_max = np.max(sv_raw)

  sv = (2 * (sv_raw - shap_min) / (shap_max - shap_min)) - 1

  custom_text = [[f"Scaled Value={sv[i][j][0][0]:.2f}. This region is interpreted to have a ({'larger' if abs(sv[i][j][0][0]) >= 1 else 'smaller'}) ({'positive' if sv[i][j][0][0]*100000 >= 0 else 'negative'}) impact on this classification result."
                for j in range(sv[:,:,0,0].shape[1])] for i in range(sv[:,:,0,0].shape[0])]

  print("Got here starting plot")

  # plot basic image
  fig = go.Figure()

  fig.add_trace(go.Image(z=img, hoverinfo="none"))

  # Parameters
  H, W, _, _ = sv.shape
  h_blocks, w_blocks = H // block_size, W // block_size  # should be 14x14

  # Compute block averages
  coarse = sv[:h_blocks*block_size, :w_blocks*block_size, 0, 0].reshape(
      h_blocks, block_size, w_blocks, block_size
  ).mean(axis=(1, 3))  # shape (14, 14)

  coarse_resized = resize(coarse, (H, W), order=0, preserve_range=True, anti_aliasing=False)

  fig.add_trace(go.Heatmap(
      z=coarse_resized,
      colorscale="RdBu",
      showscale=True,
      opacity=0.6,
      text=custom_text,
      hoverinfo="text"
  ))

  # Layout tweaks
  fig.update_layout(
      title="Explanation Map (SHAP)" + " classifies this as " + out[0],
      xaxis=dict(showticklabels=False),
      yaxis=dict(showticklabels=False)
  )


  fig.update_yaxes(autorange="reversed")

  fig.show()

  print("Got through plotting")

  return coarse_resized, fig


In [6]:
import plotly.graph_objects as go
from skimage.transform import resize

def create_interactive_explanation():
  model_name = input("Model name:")
  map = input("Map: ")
  region_size = int(input("Region size: "))
  model = load_clip('ViT-B/32')
  return model

def graph_interactive_explanation(model):
  # Todo: dont hardcode these
  _, fig = plot_shap(model, 'shap', 16)
  return fig

In [7]:
def create_and_graph_shap():
  m = create_interactive_explanation()
  fig = graph_interactive_explanation(m)
  return m, fig

In [ ]:
m, fig = create_and_graph_shap()
fig.show()

Model name:clip
Map: shap
Region size: 16


Part II: Activation-based Clickable Interface

In [ ]:
_, _, _, X = m
def preprocess_imagenet50_for_clip(X):
    tensors = []
    print(len(X))
    for i in range(len(X)):
        img = Image.fromarray(X[i].astype(np.uint8))
        tensors.append(preprocess(img))
    return torch.stack(tensors)

In [ ]:
from PIL import Image
from matplotlib import pyplot as plt
import cv2


def show_activation_interface(model):
  model, f, class_names, X = model
  X_clip = preprocess_imagenet50_for_clip(X).to(device)

  image = X_clip[1].unsqueeze(0).to(device)

  #image = preprocess(img).unsqueeze(0).to(device)

  # ---- Hook to capture all attention maps ----
  attn_maps = {}

  def get_attention_hook(layer_idx):
      def hook(module, input, output):
          attn_maps[layer_idx] = output[0].detach().cpu()
      return hook

  handles = []
  for i, block in enumerate(model.visual.transformer.resblocks):
      h = block.attn.register_forward_hook(get_attention_hook(i))
      handles.append(h)

  # ---- Forward pass ----
  with torch.no_grad():
      _ = model.encode_image(image)

  # ---- Remove hooks ----
  for h in handles:
      h.remove()

  # ---- Process attention maps ----
  def get_cls_attention_map(attn):
      # attn: (heads, tokens, tokens)
      attn_mean = attn.mean(dim=2)
      cls_attn = attn_mean[1:]  # CLS token attention to image patches
      grid_size = int(cls_attn.shape[0] ** 0.5)
      cls_attn = cls_attn.reshape(grid_size, grid_size)
      cls_attn -= cls_attn.min()
      cls_attn /= cls_attn.max()
      return cls_attn.numpy()

  cls_maps = [get_cls_attention_map(attn_maps[i]) for i in sorted(attn_maps.keys())]
  mpl_figs = []

  # ---- Overlay function ----
  def overlay_attention(img, attn_map, alpha=0.5):
      attn_resized = cv2.resize(attn_map.astype('float32'), (224, 224))
      mpl_figs.append(attn_resized)
      heatmap = plt.cm.jet(attn_resized)[..., :3]
      overlay = (1 - alpha) * np.array(img) / 255.0 + alpha * heatmap
      overlay /= overlay.max()
      return overlay

  # ---- Display progression ----
  #fig, ax = plt.subplots(figsize=(4, 4))
  for i, attn_map in enumerate(cls_maps):
      overlay = overlay_attention(X[1], attn_map)
      plt.figure()
      fig, ax = plt.subplots()
      ax.imshow(overlay)
      ax.set_title(f"Layer {i+1}")
      ax.axis('off')
      #plt.pause(0.5)

  plt.show()

  return mpl_figs, X[1], image

In [ ]:
twelve_blocks, raw_img, img = show_activation_interface(m)

In [ ]:
# plt.figure()
# plt.subplots(3, 4)
# mpl_figs = []

# for i in range(12):
#   attn_map = twelve_blocks[i]
#   mean_attn = attn_map.mean(dim=2)  # [num_tokens, num_tokens]
#   img_attn = mean_attn[1:].reshape(7,7).cpu().numpy()  # depends on ViT patch size
#   mpl_figs.append(cv2.resize(img_attn, (224, 224), interpolation=cv2.INTER_NEAREST))

#   plt.subplot(3, 4, i + 1)
#   plt.imshow(img.astype(np.uint8))
#   plt.imshow(mpl_figs[i], cmap='hot', alpha=0.5)

#   plt.title("Mean map (block " + str(i) + ")", fontsize=8)

In [ ]:
# import cv2

# for i in range(len(mpl_figs)):

#   # Original image is 224x224
#   # Explanation layers are each 7x7
#   # Result should be the explanation layers scaled up to match the dimensions of the original image

#   mpl_figs[i] = cv2.resize(mpl_figs[i], (224, 224), interpolation=cv2.INTER_NEAREST)
#   plt.figure()
#   plt.imshow(mpl_figs[i])


Dash App Functionality

TODO:
 -
 -
 -

In [ ]:
# Install required packages in Colab
!pip install dash plotly -q

import dash
from dash import dcc, html, Input, Output
import plotly.graph_objects as go
import numpy as np
from PIL import Image
import io
import base64

# # Function to convert image array to base64 for Plotly
# def array_to_base64(img_array):
#     plt.figure()
#     plt.imshow(img_array, cmap='hot')
#     """Convert numpy array (image data) to base64 string"""
#     # Normalize the array if needed
#     if img_array.dtype != np.uint8:
#         print("Normalizing")
#         print(img_array[0][0])
#         # Normalize to 0-255 range if it's float
#         if img_array.max() <= 1.0:
#             print("float")
#             img_array = (img_array * 255).astype(np.uint8)
#             print(img_array[0][0])
#             plt.figure()
#             plt.imshow(img_array, cmap='hot')
#         else:
#             print("int")
#             img_array = img_array.astype(np.uint8)

#     # Convert to PIL Image
#     if len(img_array.shape) == 2:  # Grayscale
#         print("Grayscale??")
#         img = Image.fromarray(img_array, mode='L')
#     else:  # RGB or RGBA
#         img = Image.fromarray(img_array)

#     # Convert to base64
#     buf = io.BytesIO()
#     img.save(buf, format='PNG')
#     buf.seek(0)
#     img_str = base64.b64encode(buf.read()).decode()
#     buf.close()

#     return f"data:image/png;base64,{img_str}"

IMAGE_ARRAYS = twelve_blocks

# Optional: Labels for each point
POINT_LABELS = [f'Image {i+1}' for i in range(12)]

# Initialize the Dash app
app = dash.Dash(__name__)

# Create the main line plot with 12 dots
main_x = list(range(12))
main_y = [5] * 12

main_fig = go.Figure(
    go.Scatter(
        x=main_x,
        y=main_y,
        mode='markers+lines',
        marker=dict(
            size=20,
            color='rgb(55, 128, 191)',
            line=dict(width=2, color='white')
        ),
        line=dict(color='rgb(55, 128, 191)', width=3),
        hovertemplate='<b>Point %{x}</b><extra></extra>',
        customdata=POINT_LABELS
    )
)

main_fig.update_layout(
    title='Hover over any dot to see the attention map for its layer',
    xaxis_title='Point Index',
    yaxis=dict(showticklabels=False, range=[0, 10]),
    height=250,
    margin=dict(t=50, b=30, l=50, r=30),
    hovermode='closest'
)

# Create initial image display (Point 0)
initial_image = IMAGE_ARRAYS[0] # array_to_base64()

initial_fig = go.Figure()
initial_fig.add_trace(go.Image(z=raw_img))
initial_fig.add_trace(go.Heatmap(
      z=initial_image,
      colorscale="RdBu",
      showscale=True,
      opacity=0.6
  ))
initial_fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False)
initial_fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False)
initial_fig.update_layout(
    title=f'Point 0: {POINT_LABELS[0]}',
    height=600,
    margin=dict(t=50, b=20, l=20, r=20),
)

# App layout
app.layout = html.Div([
    html.H1('Clip Interactive',
            style={'textAlign': 'center', 'marginTop': 20}),
    dcc.Graph(id='main-line', figure=main_fig),
    dcc.Graph(id='hover-graph', figure=initial_fig, config={'displayModeBar': False}),
])

# Callback to update the hover graph
@app.callback(
    Output('hover-graph', 'figure'),
    Input('main-line', 'hoverData')
)
def update_hover_graph(hover_data):
    if hover_data is None:
        # Default to point 0
        point_index = 0
    else:
        point_index = hover_data['points'][0]['x']

    # Convert the image array to base64
    img_base64 = go.Image(z=raw_img) # array_to_base64(IMAGE_ARRAYS[point_index])

    # Create Plotly figure with the image
    fig = go.Figure()
    fig.add_trace(img_base64)
    fig.add_trace(go.Heatmap(
        z=IMAGE_ARRAYS[point_index],
        colorscale="RdBu",
        showscale=True,
        opacity=0.6
    ))

    fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False)
    fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False)
    fig.update_layout(
        title=f'Point {point_index}: {POINT_LABELS[point_index]}',
        height=600,
        margin=dict(t=50, b=20, l=20, r=20),
    )

    return fig

# Run the app in Colab
if __name__ == '__main__':
    # Use Colab's built-in method to run Dash
    from google.colab import output
    output.serve_kernel_port_as_window(8050)
    app.run(mode='inline', port=8050, debug=False)